## 전처리는 왜 필요할까요?
### 많은 이유들이 있을테지만
- 결론만 말하자면, 데이터를 분석 가능한 형태로 만들어주기 위함입니다.
- 그러한 모든 과정을 전처리라 할 수 있습니다.

### 아래의 예시를 보시죠

In [ ]:
import pandas as pd
birth_dat = pd.DataFrame({
    'name': ['John', 'Anna', 'Peter', 'Linda'],
    'birth': ["1998-01-02", "980102", "19980102", "98-01-02"],
    'target_data': ['we', 'don\'t', 'know', 'yet']
})

birth_dat

In [ ]:
birth_dat['birth'] = pd.to_datetime(birth_dat['birth'])

### Pandas, Numpy

In [ ]:
import pandas as pd
import numpy as np

# 1. 결측치가 있는 데이터

In [ ]:
df = pd.DataFrame({
    '창원': [10,15 , np.nan,20,23],
    '울산': [11, np.nan, np.nan,21,24],
    '대구': [12, 17, 18, 22, 25]
})
df.index = ['1월', '2월', '3월', '4월', '5월']
df

## 결측치 확인

In [ ]:
df.isnull()

## 결측치 제거

In [ ]:
df_drop_na = df.dropna()
df_drop_na

In [ ]:
df_missing = df.fillna("missing")
df_missing

In [ ]:
df_mean = df.fillna(df.mean())
df_mean

In [ ]:
df_ffill = df.fillna(method='ffill')
df_ffill

In [ ]:
df_bfill = df.fillna(method='bfill')
df_bfill

# 2. 실제 데이터로 전처리 해보기

## 데이터 불러오기
### 전 세계 교통사고 데이터를 사용해보겠습니다.

In [ ]:
df = pd.read_csv('global_traffic_accidents.csv')
df.head()

## 데이터 둘러보기
### 데이터를 본격적으로 다루기 전에, 어떤 모습인지 먼저 살펴봅시다.

In [ ]:
# 데이터의 크기 (행 개수, 열 개수)
df.shape

In [ ]:
# 각 컬럼의 데이터 타입과 결측치 정보
df.info()

In [ ]:
# 숫자형 컬럼의 기본 통계 (평균, 최댓값, 최솟값 등)
df.describe()

In [ ]:
# 중복 데이터가 있는지 확인
df.duplicated().sum()

# 3. 새로운 컬럼 만들기
### 데이터를 분석할 때, 기존 컬럼만으로는 부족할 때가 많습니다.
### 우리가 알고 싶은 것에 맞춰 **새로운 컬럼을 직접 만들어주는 것**도 전처리의 중요한 부분입니다.

### 예를 들어, 이 데이터에는 `Vehicles Involved`(사고에 연루된 차량 수)와 `Casualties`(사상자 수)가 있지만,
### "차량 1대당 평균 몇 명의 사상자가 나왔는지" 같은 정보는 없습니다.
### 이런 정보가 필요하다면? **직접 만들면 됩니다!**

## 3-1. 계산으로 새 컬럼 만들기

In [ ]:
# 사고 규모 = 연루된 차량 수 + 사상자 수
df['Total_Impact'] = df['Vehicles Involved'] + df['Casualties']
df.head()

In [ ]:
# 차량 1대당 사상자 수
df['Casualties_per_Vehicle'] = df['Casualties'] / df['Vehicles Involved']
df.head()

## 3-2. 조건에 따라 새 컬럼 만들기
### 사상자가 5명 이상이면 '심각', 아니면 '경미'로 분류해봅시다.

In [ ]:
# np.where(조건, 참일 때 값, 거짓일 때 값)
df['Severity'] = np.where(df['Casualties'] >= 5, '심각', '경미')
df.head()

In [ ]:
# 심각/경미 사고는 각각 몇 건일까요?
df['Severity'].value_counts()

## 3-3. 문자열에서 새 컬럼 만들기
### `Location` 컬럼은 "Mumbai, India" 처럼 "도시, 국가" 형태로 되어 있습니다.
### 여기서 **국가만** 따로 빼서 새 컬럼으로 만들어봅시다.

In [ ]:
# str.split(', ')로 쉼표 기준으로 나눈 뒤, 두 번째([1]) 값만 가져오기
df['Country'] = df['Location'].str.split(', ').str[1]
df.head()

In [ ]:
# 도시도 같은 방법으로 추출 (첫 번째[0] 값)
df['City'] = df['Location'].str.split(', ').str[0]
df.head()

# 4. 조건으로 데이터 골라보기
### 새로 만든 컬럼들을 가지고, 원하는 데이터만 골라봅시다.

In [ ]:
# 심각한 사고만 보기
df[df['Severity'] == '심각'].head()

In [ ]:
# 일본에서 일어난 사고만 보기
df[df['Country'] == 'Japan'].head()

In [ ]:
# 두 조건을 동시에! 일본에서 일어난 심각한 사고
df[(df['Country'] == 'Japan') & (df['Severity'] == '심각')].head()

# 5. groupby로 묶어서 보기
### `groupby`는 "같은 값끼리 묶어서 한 번에 계산하기" 라고 생각하면 쉽습니다.
### 예: 국가별로 사고 건수, 원인별 평균 사상자 수 등

In [ ]:
# 국가별 사고 건수
df.groupby('Country').size()

In [ ]:
# 사고 원인별 평균 사상자 수
df.groupby('Cause')['Casualties'].mean()

In [ ]:
# 날씨별 사고 건수
df.groupby('Weather Condition').size()

# 6. 새로 만든 데이터프레임 저장하기
### 우리가 컬럼을 추가한 데이터프레임을 csv 파일로 저장해봅시다.

In [ ]:
df.to_csv('modified_global_traffic_accidents.csv', index=False)

# 실습 과제
## 1. `Casualties`가 8명 이상이면 '대형사고', 그렇지 않으면 '일반사고'로 분류하는 `Accident_Size` 컬럼을 만들어보세요.
## 2. 각 국가(`Country`)별로 평균 사상자 수(`Casualties`)를 구해보세요.
## 3. 날씨(`Weather Condition`)가 'Clear'가 아닌 사고만 골라서, 그 중 사상자가 가장 많이 발생한 원인(`Cause`)이 무엇인지 알아보세요.
## 4. 위에서 새로 만든 컬럼들이 모두 포함된 데이터프레임을 `my_traffic_data.csv`로 저장해보세요.